If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec33_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 33: Regression Inference
v.ekc

Our sample gives us *a* regression line — but it's one draw from a noisy world. Today: the regression **model** (signal + noise), bootstrap confidence intervals for predictions and for the true slope, and a hypothesis test for "is there really a line at all?" (Slides: KL Lecture 33 deck.)

**Today's sections:**
1. Residuals, visualized
2. The regression model
3. Prediction at a given x
4. Bootstrapping predictions
5. Inference for the true slope

In [ ]:
from datascience import *
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

In [ ]:
def standard_units(arr):
    return (arr - np.average(arr))/np.std(arr)

def correlation(t, x, y):
    x_standard = standard_units(t.column(x))
    y_standard = standard_units(t.column(y))
    return np.average(x_standard * y_standard)

def slope(t, x, y):
    r = correlation(t, x, y)
    y_sd = np.std(t.column(y))
    x_sd = np.std(t.column(x))
    return r * y_sd / x_sd

def intercept(t, x, y):
    x_mean = np.mean(t.column(x))
    y_mean = np.mean(t.column(y))
    return y_mean - slope(t, x, y)*x_mean

def fitted_values(t, x, y):
    """Return an array of the regression estimates at all the x values"""
    a = slope(t, x, y)
    b = intercept(t, x, y)
    return a*t.column(x) + b

def residuals(t, x, y):
    """Return an array of all the residuals"""
    predictions = fitted_values(t, x, y)
    return t.column(y) - predictions

---
## 1. Residuals, Visualized

Warm-up from last time: residual plots for the regression line, two bad lines, and the US women curve. (Deck DQs: KL Lecture 33, slides 9–10.)

In [ ]:
demographics = Table.read_table('district_demographics2016.csv')
demographics = demographics.drop(
    'State', 'District', 'Percent voting for Clinton')

slope(demographics, 'College%', 'Median Income'), intercept(demographics, 'College%', 'Median Income')

In [ ]:
def plot_residuals_for_given_params(t, x, y, a, b):
    x_arr = t.column(x)
    predictions = a*x_arr + b
    tbl = t.with_columns(
        'Estimate', predictions,
        'Residual', t.column(y) - predictions
    )
    tbl.select(x, y, 'Estimate').scatter(0)
    tbl.scatter(x, 'Residual')
    avg_resid = np.round(np.mean(t.column(y) - predictions))
    corr_resid = np.round(correlation(tbl,x,'Residual'),2)
    print("Mean: " + str(avg_resid) + "    Correlation with x: " + str(corr_resid))

In [ ]:
plot_residuals_for_given_params(demographics, 
                                'College%', 'Median Income', 
                                slope(demographics, 'College%', 'Median Income'), 
                                intercept(demographics, 'College%', 'Median Income'))

In [ ]:
# 1 
plot_residuals_for_given_params(demographics, 'College%', 'Median Income', 1270, 10802)

In [ ]:
# 2
plot_residuals_for_given_params(demographics, 'College%', 'Median Income', 700, 38431)

In [ ]:
# 3
us_women = Table.read_table('us_women.csv')
plot_residuals_for_given_params(us_women, 'ave weight', 'height', 
                                slope(us_women,'ave weight', 'height'),
                                intercept(us_women,'ave weight', 'height'))

---
## 2. The Regression Model

**Model:** the truth is a perfect line (*signal*), and each observation is that line plus random *noise*. We only see the noisy dots — so our fitted line is an *estimate* of the hidden true line. Run these several times: small samples miss badly, big samples hug the truth. (KL deck, slides 11–16.)

In [ ]:
# Ignore this code; it's graphics for demonstrating the regression model
def draw_and_compare(true_slope, true_int, sample_size):
    x = np.random.normal(50, 5, sample_size)
    xlims = np.array([np.min(x), np.max(x)])
    errors = np.random.normal(0, 6, sample_size)
    y = (true_slope * x + true_int) + errors
    sample = Table().with_columns('x', x, 'y', y)

    sample.scatter('x', 'y')
    plots.plot(xlims, true_slope*xlims + true_int, lw=2, color='green')
    plots.title('True Line, and Points Created')

    sample.scatter('x', 'y')
    plots.title('What We Get to See')

    sample.scatter('x', 'y', fit_line=True)
    plots.title('Regression Line: Estimate of True Line')

    sample.scatter('x', 'y', fit_line=True)
    plots.plot(xlims, true_slope*xlims + true_int, lw=2, color='green')
    plots.title("Regression Line and True Line")

In [ ]:
draw_and_compare(2, -5, 10)

In [ ]:
draw_and_compare(2, -5, 10)

In [ ]:
draw_and_compare(2, -5, 100)

In [ ]:
draw_and_compare(2, -5, 100)

---
## 3. Prediction at a Given x

Birth weight vs. gestational days. Our line predicts ≈ 129.2 oz at 300 days — but a different sample would predict differently. How much could it vary?

In [ ]:
births = Table.read_table('baby.csv')
births.show(3)

In [ ]:
births.scatter('Gestational Days', 'Birth Weight')

In [ ]:
births = births.where('Gestational Days', are.between(225, 325))

In [ ]:
births.scatter('Gestational Days', 'Birth Weight')

## Suppose we assume the regression model

In [ ]:
correlation(births, 'Gestational Days', 'Birth Weight')

In [ ]:
births.scatter('Gestational Days', 'Birth Weight', fit_line=True)

## Prediction at a Given Value of x

In [ ]:
def prediction_at(t, x, y, x_value):
    '''
    t - table
    x - label of x column
    y - label of y column
    x_value - the x value for which we want to predict y
    '''
    return slope(t, x, y) * x_value + intercept(t, x, y)

In [ ]:
prediction_at_300 = prediction_at(births, 'Gestational Days', 'Birth Weight', 300)
prediction_at_300

In [ ]:
x = 300
births.scatter('Gestational Days', 'Birth Weight', fit_line=True)
plots.plot([x, x], [40, prediction_at_300], color='red', lw=2);

---
## 4. Bootstrapping Predictions

Familiar move, new target: **resample the scatter, refit the line, re-predict** — the spread of re-predictions is our uncertainty.

In [ ]:
# You don't need to understand the plotting code in this cell,
# but you should understand the figure that comes out.

plots.figure(figsize=(10, 11))
plots.subplot(3, 2, 1)
plots.scatter(births[1], births[0], s=10, color='darkblue')
plots.xlim([225, 325])
plots.title('Original sample')

for i in np.arange(1, 6, 1):
    plots.subplot(3,2,i+1)
    resampled = births.sample()
    plots.scatter(resampled.column('Gestational Days'), resampled.column('Birth Weight'), s=10, color='tab:green')
    plots.xlim([225, 325])
    plots.title('Bootstrap sample '+str(i))
plots.tight_layout()

In [ ]:
for i in np.arange(4):
    resample = births.sample()
    predicted_y = prediction_at(resample, 'Gestational Days', 'Birth Weight', 300)
    print('Predicted y from bootstramp sample was', predicted_y)
    resample.scatter('Gestational Days', 'Birth Weight', fit_line=True)
    plots.scatter(300, predicted_y, color='gold', s=50, zorder=3);
    plots.plot([x, x], [40, predicted_y], color='red', lw=2);
    plots.plot([200, x], [predicted_y, predicted_y], color='red', lw=2);

In [ ]:
lines = Table(['slope','intercept', 'at 210', 'at 300', 'at 320'])

for i in range(10):
    resample = births.sample()
    a = slope(resample, 'Gestational Days', 'Birth Weight')
    b = intercept(resample, 'Gestational Days', 'Birth Weight')
    lines.append([a, b, a * 210 + b, a * 300 + b, a * 320 + b])

for i in np.arange(lines.num_rows):
    line = lines.row(i)
    plots.plot([210, 320], [line.item('at 210'), line.item('at 320')], lw=1)
    plots.scatter(300, line.item('at 300'), s=30, zorder=3)

In [ ]:
np.mean(births.column('Gestational Days')), np.mean(births.column('Birth Weight'))

In [ ]:
lines = Table(['slope','intercept', 'at 291', 'at 300', 'at 309'])

for i in range(10):
    resample = births.sample()
    a = slope(resample, 'Gestational Days', 'Birth Weight')
    b = intercept(resample, 'Gestational Days', 'Birth Weight')
    lines.append([a, b, a * 291 + b, a * 300 + b, a * 309 + b])
lines


In [ ]:
for i in np.arange(lines.num_rows):
    line = lines.row(i)
    plots.plot([291, 309], [line.item('at 291'), line.item('at 309')], lw=1)
    plots.scatter(300, line.item('at 300'), s=30, zorder=3)

### Prediction Interval

In [ ]:
def bootstrap_prediction(t, x, y, new_x, repetitions=2500):
    """ 
    Makes a 95% confidence interval for the height of the true line at new_x, 
    using linear regression on the data in t (column names x and y).
    Shows a histogram of the bootstrap samples and shows the interval
    in gold.
    """

    # Bootstrap the scatter, predict, collect
    predictions = make_array()
    for i in np.arange(repetitions):
        resample = t.sample() #bootstrap
        predicted_y = prediction_at(resample,x,y,new_x) #get prediction at new_x using the resample data
        predictions = np.append(predictions, predicted_y) #add prediction to the predictions array

    # Find the ends of the approximate 95% prediction interval
    left = percentile(2.5, predictions) # get lower bound of confidence interval
    right = percentile(97.5, predictions) # get upper bound of confidence interval
    round_left = round(left, 3)
    round_right = round(right, 3)

    # Display results
    Table().with_column('Prediction', predictions).hist(bins=20)
    plots.xlabel('predictions at x='+str(new_x))
    plots.plot([left, right], [0, 0], color='yellow', lw=8);
    print('Approximate 95%-confidence interval for height of true line at x =', new_x)
    print(round_left, 'to', round_right, '( width =', round(right - left, 3), ')') 

In [ ]:
bootstrap_prediction(births, 'Gestational Days', 'Birth Weight', 300)

### Predictions at Different Values of x

In [ ]:
bootstrap_prediction(births, 'Gestational Days', 'Birth Weight', 230)

In [ ]:
bootstrap_prediction(births, 'Gestational Days', 'Birth Weight', 280)

In [ ]:
# No need to follow the code in this cell; just understand the graph

lines = Table(['slope','intercept', 'at 210', 'at 300', 'at 320'])

for i in range(10):
    resample = births.sample()
    a = slope(resample, 'Gestational Days', 'Birth Weight')
    b = intercept(resample, 'Gestational Days', 'Birth Weight')
    lines.append([a, b, a * 210 + b, a * 300 + b, a * 320 + b])

for i in np.arange(lines.num_rows):
    line = lines.row(i)
    plots.plot([210, 320], [line.item('at 210'), line.item('at 320')], lw=1)

> **Notice the widths:** intervals are narrowest near the center of the data and widen at the edges — the bootstrap lines all pivot near the middle, so they fan out at the extremes. Extrapolation is risky, and now you can see why.

---
## 5. Inference for the True Slope

Same recipe, statistic = slope. And the payoff question: **is the slope real, or could it be 0?**

In [ ]:
births.scatter('Gestational Days', 'Birth Weight', fit_line=True)

In [ ]:
slope(births, 'Gestational Days', 'Birth Weight')

In [ ]:
def bootstrap_slope(t, x, y, repetitions=2500):
    """ 
    Makes a 95% confidence interval for the slope of the true line, 
    using linear regression on the data in t (column names x and y).
    Shows a histogram of the bootstrap samples and shows the interval
    in gold.
    """
    
    # Bootstrap the scatter, find the slope, collect
    slopes = make_array()
    for i in np.arange(repetitions):
        bootstrap_sample = t.sample()
        bootstrap_slope = slope(bootstrap_sample, x, y)
        slopes = np.append(slopes, bootstrap_slope)
    
    # Find the endpoints of the 95% confidence interval for the true slope
    left = percentile(2.5, slopes)
    right = percentile(97.5, slopes)
    round_left = round(left, 3)
    round_right = round(right, 3)
    
    # Slope of the regression line from the original sample
    observed_slope = slope(t, x, y)
    
    # Display results (no need to follow this code)
    Table().with_column('Bootstrap Slopes', slopes).hist(bins=20)
    plots.plot(make_array(left, right), make_array(0, 0), color='yellow', lw=8);
    print('Slope of regression line:', round(observed_slope, 3))
    print('Approximate 95%-confidence interval for the slope of the true line:')
    print(round_left, 'to', round_right)

In [ ]:
bootstrap_slope(births, 'Gestational Days', 'Birth Weight')

### Rain on the Regression Parade ☔

In [ ]:
draw_and_compare(0, 10, 25)

### Maternal Age and Birth Weight

In [ ]:
births.scatter('Maternal Age', 'Birth Weight')

In [ ]:
slope(births, 'Maternal Age', 'Birth Weight')

In [ ]:
births.scatter('Maternal Age', 'Birth Weight', fit_line=True)

**Null:** Slope of true line is equal to 0.

**Alternative:** Slope of true line is not equal to 0.

In [ ]:
bootstrap_slope(births, 'Maternal Age', 'Birth Weight')

> **Read the interval:** the 95% CI for the maternal-age slope runs from below 0 to above 0 — it *contains* 0, so we can't reject the null. The tiny observed slope (≈ 0.09) is entirely consistent with "no true relationship." Compare gestational days, whose interval sat far from 0.

---
> **Today's takeaway:** the fitted line is an estimate; bootstrap it to get CIs for predictions and slope; and "does the CI contain 0?" turns regression into a hypothesis test. This is Homework 11, start to finish.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — slope inference
```python
def bootstrap_slope_ci(t, x, y, reps=2500):
    slopes = make_array()
    for i in np.arange(reps):
        slopes = np.append(slopes, slope(t.sample(), x, y))
    return percentile(2.5, slopes), percentile(97.5, slopes)
# CI contains 0  =>  can't reject "no true slope"
```